# Module 11 Guided Lab: Final Data Checks, Data Dictionary, and ABT Documentation

**BAN 6003: Data Management and Analytics Integration**

This module is the final preparation step before we apply analytics to an ABT.

We are not learning many new Pandas or SQL functions this week. Instead, we are using the skills we already learned to ask:

> Is this ABT ready for modeling, handoff, and business interpretation?

A good ABT should be checked, documented, and understandable to someone other than the person who built it.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Conduct final sanity checks on a prepared ABT.
2. Check missing values, impossible values, duplicate entity rows, and target variable issues.
3. Create a simple data dictionary for an ABT.
4. Write a transformation and integration summary.
5. Prepare a data handoff package.
6. Apply this same process to your semester project ABT.

Starting next module, we will apply analytics and modeling. That means this is the moment to slow down and make sure the ABT is trustworthy.

## Business Scenario

Imagine you are handing your ABT to another analyst or to your future self.

That person needs to understand:

- what one row means
- where the columns came from
- what transformations were applied
- what the target variable means
- what missing values or limitations still exist
- whether the table is safe to use for modeling

If they cannot understand these points, the ABT is not ready for handoff.

## 0. Setup

This lab uses a small demonstration ABT that the notebook builds from `nycflights13`. You do not need to load any other data.

For the demo, we will build a route-level ABT with one row per origin-destination route.

After the lab, apply the same documentation steps to your own project ABT in your project repository. That work is separate from this notebook.

### Package setup note

You can work in GitHub Codespaces or in local VS Code. On the UTSA campus network Codespaces may not connect; use the local setup there (see the Canvas page on local setup).

If you are using **GitHub Codespaces**, the required packages should already be installed from `requirements.txt` when the Codespace is created. You usually do not need to run any `%pip install` command.

If you are using **local VS Code** and an import fails, uncomment and run the `%pip install` line(s) in the setup cell below, then rerun the imports.


In [3]:
# Codespaces: nycflights13 is installed from requirements.txt.
# Local only: if the import below fails, uncomment and run this line:
# %pip install nycflights13 "setuptools<81"
# (setuptools<81 is needed because nycflights13 uses pkg_resources, which newer setuptools removed.)

import nycflights13
import pandas as pd
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)


/Users/jh/Documents/GitHub/ban6003-week-11-abt_documentation/.venv/lib/python3.14/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


In [4]:
flights = nycflights13.flights.copy()
airports = nycflights13.airports.copy()
airlines = nycflights13.airlines.copy()

## 1. Build a Demo ABT

For this demonstration, we will create a route-level ABT.

**Unit of analysis:** one row per origin-destination route.

This ABT is not meant to be perfect. It is a realistic practice table for final checks and documentation.

In [5]:
route_abt = (
    flights
    .groupby(["origin", "dest"])
    .agg(
        flight_count=("flight", "count"),
        avg_dep_delay=("dep_delay", "mean"),
        avg_arr_delay=("arr_delay", "mean"),
        max_arr_delay=("arr_delay", "max"),
        avg_distance=("distance", "mean"),
        avg_air_time=("air_time", "mean")
    )
    .reset_index()
)

route_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time
0,EWR,ALB,439,23.620525,14.397129,328.0,143.0,31.787081
1,EWR,ANC,8,12.875000,-2.500000,39.0,3370.0,413.125000
2,EWR,ATL,5022,15.501738,13.233183,796.0,746.0,111.993847
3,EWR,AUS,968,11.508863,-0.474399,349.0,1504.0,211.247649
4,EWR,AVL,265,8.616601,8.804781,228.0,583.0,89.796813


Add destination airport name to make the table more understandable.

This is a left merge because the route-level ABT is our main table and we want to preserve its rows.

In [6]:
dest_airports = airports[["faa", "name", "lat", "lon"]].rename(
    columns={
        "faa": "dest",
        "name": "dest_airport_name",
        "lat": "dest_lat",
        "lon": "dest_lon"
    }
)

route_abt = route_abt.merge(dest_airports, how="left", on="dest")

route_abt.head()

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_airport_name,dest_lat,dest_lon
0,EWR,ALB,439,23.620525,14.397129,328.0,143.0,31.787081,Albany Intl,42.748267,-73.801692
1,EWR,ANC,8,12.875000,-2.500000,39.0,3370.0,413.125000,Ted Stevens Anchorage Intl,61.174361,-149.996361
2,EWR,ATL,5022,15.501738,13.233183,796.0,746.0,111.993847,Hartsfield Jackson Atlanta Intl,33.636719,-84.428067
3,EWR,AUS,968,11.508863,-0.474399,349.0,1504.0,211.247649,Austin Bergstrom Intl,30.194528,-97.669889
4,EWR,AVL,265,8.616601,8.804781,228.0,583.0,89.796813,Asheville Regional Airport,35.436194,-82.541806


## 2. Final Sanity Check 1: Row Meaning and Shape

Before checking values, first confirm what one row means.

For this ABT:

> One row should represent one origin-destination route.

The key columns should be `origin` and `dest`.

In [7]:
route_abt.shape

(224, 11)

In [8]:
route_abt[["origin", "dest"]].head()

,origin,dest
0,EWR,ALB
1,EWR,ANC
2,EWR,ATL
3,EWR,AUS
4,EWR,AVL


### Check duplicate entity rows

If one row should represent one route, then each origin-destination pair should appear only once.

In [9]:
route_abt[["origin", "dest"]].duplicated().sum()

np.int64(0)

If this number is greater than zero, the ABT has duplicate entity rows and should not be used for modeling until the issue is resolved.

### Your Turn 1

Cell 8 grouped `flights` by `origin` and `dest`, so one row of `route_abt` is one route. Suppose a teammate had grouped by `origin`, `dest` and `carrier` instead.

1. In the code cell below, build the teammate's table: group `flights` by `["origin", "dest", "carrier"]`, count the flights with `.size()`, and save the result as `route_carrier` with `.reset_index(name="flight_count")`. Then print the number of rows of `route_carrier` and the duplicate check on `["origin", "dest"]`.
2. Write 2 sentences: What does one row of `route_carrier` represent, and which columns are its key? Why does the duplicate check on `origin` and `dest` no longer return 0?

In [10]:
# Your Turn 1
# Build route_carrier: group flights by origin, dest and carrier, count with .size(),
# then .reset_index(name="flight_count").
# Print len(route_carrier) and the duplicate check on ["origin", "dest"] (same code as in the cell above).

route_carrier = flights.groupby(["origin", "dest", "carrier"]).size().reset_index(name="flight_count")
print(len(route_carrier))
print(route_carrier.duplicated(subset=["origin", "dest", "carrier"]).sum())


439
0


**Your answer:**  
Type your answer here. (1 or 2 sentences)

## 3. Final Sanity Check 2: Missing Values

Missing values are not always wrong, but they must be understood.

For an ABT, missing values can affect:

- modeling
- interpretation
- business recommendations
- whether a column should be used at all

In [11]:
missing_summary = route_abt.isna().sum().reset_index()
missing_summary.columns = ["column_name", "missing_count"]
missing_summary["missing_percent"] = (missing_summary["missing_count"] / len(route_abt) * 100).round(1)

missing_summary.sort_values("missing_count", ascending=False)

,column_name,missing_count,missing_percent
8,dest_airport_name,7,3.1
9,dest_lat,7,3.1
10,dest_lon,7,3.1
3,avg_dep_delay,1,0.4
4,avg_arr_delay,1,0.4
5,max_arr_delay,1,0.4
7,avg_air_time,1,0.4
0,origin,0,0.0
1,dest,0,0.0
2,flight_count,0,0.0


### Questions to ask

For any column with missing values:

- Is this missingness expected?
- Did it come from the raw data?
- Did it come from an integration step?
- Should the column be kept, dropped, or imputed later?
- Should the limitation be documented?

This module is about checking and documenting, not making every dataset perfect.

### Your Turn 2

Choose one column from the missing summary that has missing values. Do not choose the target `avg_arr_delay`; Section 5 covers it.

1. In the code cell below, show the rows where your column is missing, with `origin`, `dest`, `flight_count` and your column.
2. Answer the five questions in **Questions to ask** for your column, one short sentence per question.

In [12]:
# Your Turn 2
# Replace your_column with the column you chose, then show its missing rows.
# Hint: route_abt[route_abt["your_column"].isna()][["origin", "dest", "flight_count", "your_column"]]

route_abt[route_abt["dest_airport_name"].isna()][["origin", "dest", "flight_count", "dest_airport_name"]]

,origin,dest,flight_count,dest_airport_name
8,EWR,BQN,297,NaN
75,EWR,SJU,1067,NaN
79,EWR,STT,189,NaN
93,JFK,BQN,599,NaN
135,JFK,PSE,365,NaN
148,JFK,SJU,4752,NaN
153,JFK,STT,333,NaN


**Your answer:**  
Column: 

1. 
2. 
3. 
4. 
5. 

## 4. Final Sanity Check 3: Impossible or Suspicious Values

Some values may be technically non-missing but still suspicious.

Examples:

- negative flight counts
- negative distances
- impossible average air times
- unusually large delay values

Not every extreme value is wrong. But it should be noticed.

In [13]:
route_abt.describe()

,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_lat,dest_lon
count,224.000000,223.000000,223.000000,223.000000,224.000000,223.000000,217.000000,217.000000
mean,1503.464286,14.460038,7.803616,367.802691,987.339773,143.228092,37.151552,-89.151989
std,1851.848985,7.076796,8.766000,220.813783,778.512987,97.771737,5.787763,15.535549
min,1.000000,-9.000000,-22.000000,-22.000000,17.000000,25.466019,21.318681,-157.922428
25%,289.250000,10.102531,2.874643,250.500000,440.750000,73.305461,33.636719,-94.713905
50%,765.500000,13.664606,7.517555,334.000000,754.000000,115.372401,38.174086,-84.219375
75%,2061.000000,18.783245,12.612513,413.500000,1376.250000,195.040787,41.533972,-80.040528
max,11262.000000,41.818471,44.585106,1272.000000,4983.000000,623.087719,61.174361,-68.828139


In [14]:
# Basic impossible-value checks
checks = {
    "negative_flight_count": (route_abt["flight_count"] < 0).sum(),
    "zero_or_negative_distance": (route_abt["avg_distance"] <= 0).sum(),
    "zero_or_negative_air_time": (route_abt["avg_air_time"] <= 0).sum(),
}

checks

{'negative_flight_count': np.int64(0),
 'zero_or_negative_distance': np.int64(0),
 'zero_or_negative_air_time': np.int64(0)}

### Check extreme values

Now sort to inspect routes with very high average arrival delay.

In [15]:
route_abt.sort_values("avg_arr_delay", ascending=False).head(10)

,origin,dest,flight_count,avg_dep_delay,avg_arr_delay,max_arr_delay,avg_distance,avg_air_time,dest_airport_name,dest_lat,dest_lon
13,EWR,CAE,104,36.336842,44.585106,224.0,602.0,92.148936,Columbia Metropolitan,33.938833,-81.119528
84,EWR,TYS,323,41.818471,41.150160,281.0,631.0,97.635783,Mc Ghee Tyson,35.810972,-83.994028
82,EWR,TUL,315,34.906355,33.659864,262.0,1215.0,178.309524,Tulsa Intl,36.198389,-95.888111
53,EWR,OKC,346,30.568807,30.619048,262.0,1325.0,193.009524,Will Rogers World,35.393089,-97.600733
36,EWR,JAC,23,28.700000,29.894737,175.0,1874.0,263.263158,Jackson Hole Airport,43.607333,-110.737750
65,EWR,RIC,1667,26.478960,25.820433,463.0,277.0,53.543653,Richmond Intl,37.505167,-77.319667
47,EWR,MKE,1094,23.551174,24.003781,441.0,725.0,115.372401,General Mitchell Intl,42.947222,-87.896583
23,EWR,DSM,411,29.328244,23.548718,322.0,1017.0,149.541026,Des Moines Intl,41.533972,-93.663083
48,EWR,MSN,354,21.437143,23.197101,337.0,799.0,124.246377,Dane Co Rgnl Truax Fld,43.139858,-89.337514
63,EWR,PWM,770,25.714674,23.005479,310.0,284.0,48.727397,Portland Intl Jetport,43.646161,-70.309281


Extreme values do not automatically mean errors. They may be real business signals. But they should be reviewed before modeling.

### Your Turn 3

`describe()` shows that `max_arr_delay` has a minimum of -22 minutes. A negative *maximum* delay looks impossible.

1. In the code cell below, show the routes where `max_arr_delay` is below 0, with their `flight_count`.
2. Write 1-2 sentences: is this an error? What does it tell you about averages and maximums on routes with very few flights?

In [16]:
# Your Turn 3
# Show the routes with a negative max_arr_delay and their flight_count.
# Hint: filter route_abt with a Boolean condition, then select a few columns.


**Your answer:**  
Type your answer here.

## 5. Final Sanity Check 4: Target Variable Issues

If the ABT will be used for modeling, the target variable must be clearly defined.

For this demo, suppose an airline planning team wants to estimate, **before a new schedule season starts**, how late flights on each route will arrive on average, so it can add time buffers to the schedule.

- **Target:** `avg_arr_delay`, the average arrival delay of a route in minutes. This is the value a model would predict.
- **Predictors (features):** other route information the team already knows when it plans the schedule, such as `avg_distance`, the number of flights on the route (`flight_count`) and the destination location.

In Module 12 you will fit models like this one. This section checks whether the target is ready for it.

In [17]:
route_abt["avg_arr_delay"].isna().sum()

np.int64(1)

In [18]:
route_abt["avg_arr_delay"].describe()

count    223.000000
mean       7.803616
std        8.766000
min      -22.000000
25%        2.874643
50%        7.517555
75%       12.612513
max       44.585106
Name: avg_arr_delay, dtype: float64

### Questions to ask about the target

- What exactly does the target measure?
- Is the target missing for some rows?
- When is the target known? Only after the flights have arrived. That is why we predict it: the target is never known at the time of prediction.
- Is every predictor known at the time of prediction, before the target is known? A predictor that is not yet known then is **data leakage** (Module 12 covers this in detail).
- Are there extreme values that should be reviewed?
- Would this target support the business question?

These questions help prevent modeling mistakes later.

### Which flights does the target cover?

`flight_count` counts every flight record, including cancelled flights. `avg_arr_delay` averages only the flights that have a recorded arrival delay. The cell below compares the two for each route.

In [19]:
arr_delay_counts = (
    flights.groupby(["origin", "dest"])["arr_delay"].count()
    .rename("flights_with_arr_delay").reset_index()
)
target_coverage = route_abt[["origin", "dest", "flight_count", "avg_arr_delay"]].merge(
    arr_delay_counts, how="left", on=["origin", "dest"]
)
target_coverage["flights_without_arr_delay"] = (
    target_coverage["flight_count"] - target_coverage["flights_with_arr_delay"]
)

print("Flights without an arrival delay, all routes:", target_coverage["flights_without_arr_delay"].sum())
target_coverage.sort_values("flights_without_arr_delay", ascending=False).head()

Flights without an arrival delay, all routes: 9430


,origin,dest,flight_count,avg_arr_delay,flights_with_arr_delay,flights_without_arr_delay
204,LGA,ORD,8857,1.833431,8507,350
161,LGA,BOS,4283,-2.419540,4002,281
55,EWR,ORD,6100,8.997255,5828,272
175,LGA,DCA,4716,5.838406,4468,248
156,LGA,ATL,10263,11.322478,10041,222


### Your Turn 4

Use the coverage table above and the route in its first row.

Write 2 sentences: How many flights does `flight_count` include for this route, and how many flights does `avg_arr_delay` average? Why are the other flights left out of the target? Name the pandas behavior that leaves them out.

**Your answer:**  
Type your answer here.

### Watch for columns that are not known at the time of prediction

The planning team makes its estimate before the season starts. Two columns in this ABT are measured on the same flights as the target, so they are known only after those flights have flown:

- `max_arr_delay` is the largest of the same `arr_delay` values that the target averages.
- `avg_dep_delay` is the average departure delay of the same flights. A late departure usually means a late arrival, so it is strongly related to the target (see the correlation below).

A model that used either column as a predictor would be using information that does not exist yet when the estimate is needed. With `avg_dep_delay` the model would look accurate in the notebook and still be useless for planning.

This is not an error in the table, but the receiving analyst must know it. Record it in the data dictionary notes for these two columns.

In [20]:
route_abt[["avg_arr_delay", "avg_dep_delay", "max_arr_delay"]].corr().round(2)

,avg_arr_delay,avg_dep_delay,max_arr_delay
avg_arr_delay,1.00,0.83,0.12
avg_dep_delay,0.83,1.00,0.02
max_arr_delay,0.12,0.02,1.00


## 6. Create a Data Dictionary

A data dictionary explains what each column of the ABT means. **Each row of the data dictionary describes one ABT column**, so the route ABT with 11 columns needs 11 dictionary rows.

Each dictionary row has these fields:

- column name
- data type
- description
- source
- transformation or derivation
- notes or concerns

A data dictionary is not busywork. It is what makes your dataset usable by someone else.

In [21]:
data_dictionary = pd.DataFrame({
    "column_name": route_abt.columns,
    "data_type": [str(route_abt[col].dtype) for col in route_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": ""
})

data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,,,,
1,dest,str,,,,
2,flight_count,int64,,,,
3,avg_dep_delay,float64,,,,
4,avg_arr_delay,float64,,,,
5,max_arr_delay,float64,,,,
6,avg_distance,float64,,,,
7,avg_air_time,float64,,,,
8,dest_airport_name,str,,,,
9,dest_lat,float64,,,,


The cell above created one dictionary row per ABT column, with the four documentation fields still blank.

Next, fill in the dictionary rows for five ABT columns as examples. `doc_fields` lists the four documentation fields, and each entry gives the four values for one ABT column, in that order.

In a real project, you complete a dictionary row for every final ABT column.

In [22]:
doc_fields = ["description", "source", "transformation_or_derivation", "notes_or_concerns"]

# One entry per ABT column (= one dictionary row). The four values follow the order of doc_fields.
example_entries = {
    "origin": [
        "Origin airport code for the route", "flights table",
        "Used as part of the route-level grouping key",
        "Forms the route key together with dest; only the three New York City airports (EWR, JFK, LGA)"
    ],
    "dest": [
        "Destination airport code for the route", "flights table",
        "Used as part of the route-level grouping key",
        "Includes one unusual route, EWR to LGA, with a single cancelled flight"
    ],
    "flight_count": [
        "Number of flights observed for the route", "flights table",
        "Count of flight records grouped by origin and destination",
        "Includes cancelled flights; ranges from 1 to 11,262; 10 routes have fewer than 10 flights, so their "
        "averages are unstable"
    ],
    "avg_arr_delay": [
        "Average arrival delay for flights on the route, in minutes (target)", "flights table",
        "Mean of arr_delay grouped by origin and destination",
        "Averages only flights with a recorded arrival delay; missing for EWR to LGA (its only flight was "
        "cancelled); negative means early on average"
    ],
    "dest_airport_name": [
        "Full name of the destination airport", "airports table",
        "Joined from airports using dest = faa",
        "Missing for 7 routes whose destination (BQN, PSE, SJU, STT) is not in the airports table"
    ],
}

for abt_column, values in example_entries.items():
    data_dictionary.loc[data_dictionary["column_name"] == abt_column, doc_fields] = values

data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,Origin airport code for the route,flights table,Used as part of the route-level grouping key,Forms the route key together with dest; only t...
1,dest,str,Destination airport code for the route,flights table,Used as part of the route-level grouping key,"Includes one unusual route, EWR to LGA, with a..."
2,flight_count,int64,Number of flights observed for the route,flights table,Count of flight records grouped by origin and ...,Includes cancelled flights; ranges from 1 to 1...
3,avg_dep_delay,float64,,,,
4,avg_arr_delay,float64,Average arrival delay for flights on the route...,flights table,Mean of arr_delay grouped by origin and destin...,Averages only flights with a recorded arrival ...
5,max_arr_delay,float64,,,,
6,avg_distance,float64,,,,
7,avg_air_time,float64,,,,
8,dest_airport_name,str,Full name of the destination airport,airports table,Joined from airports using dest = faa,"Missing for 7 routes whose destination (BQN, P..."
9,dest_lat,float64,,,,


### Your Turn 5

Complete the dictionary row for the ABT column `avg_dep_delay`. Use the same pattern as the examples above: four values in the order of `doc_fields`.

In `notes_or_concerns`, record two findings from earlier sections: the route where `avg_dep_delay` is missing, and the timing concern from Section 5.

In the offline assignment you will complete the dictionary rows for the five ABT columns that are still blank.

In [23]:
# Your Turn 5 - document the ABT column avg_dep_delay (one dictionary row).
# Replace the four empty strings, in the order of doc_fields, then run the cell.
your_turn_entries = {
    "avg_dep_delay": ["", "", "", ""],
}

for abt_column, values in your_turn_entries.items():
    data_dictionary.loc[data_dictionary["column_name"] == abt_column, doc_fields] = values

data_dictionary[data_dictionary["column_name"] == "avg_dep_delay"]

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
3,avg_dep_delay,float64,,,,


## 7. Transformation and Integration Summary

The data dictionary documents columns. The transformation summary documents the process.

A good transformation summary answers:

- What raw data was used?
- What cleaning or filtering was applied?
- What transformations were created?
- What aggregations changed the unit of analysis?
- What tables were joined?
- What checks were performed?
- What assumptions or limitations remain?

### Example Transformation Summary

For this demo ABT (this example is deliberately brief so that it fits in the lab; a project summary is usually longer and more detailed, with row counts before and after each step, the reason for each transformation, and the result of each check):

1. Started with the `flights` table from `nycflights13`.
2. Aggregated flight-level records to one row per origin-destination route.
3. Calculated flight count, average departure delay, average arrival delay, maximum arrival delay, average distance, and average air time.
4. Joined destination airport name and location from the `airports` table using `dest = faa`.
5. Checked duplicate route keys using `origin` and `dest`.
6. Checked missing values across ABT columns.
7. Reviewed basic impossible values and extreme delay values.
8. The target variable for later modeling may be `avg_arr_delay`, but this choice should be justified by the business question.

### Your Turn 6

The example summary says which checks were performed, but not what they found. The receiving analyst needs both.

Add a short **Findings and limitations** list (3-4 bullets) to the demo summary. Use numbers from Sections 2-5 and from Your Turns 2-4, for example duplicate keys, missing values, very small routes, and what the target covers.

**Findings and limitations:**  
- Type your first bullet here.

## 8. Prepare a Handoff Package

A complete ABT handoff package usually has five parts:

1. Final ABT file
2. Complete data dictionary
3. Transformation and integration summary
4. Validation checklist
5. README or short handoff note

For your semester project, the final ABT and the data dictionary are required files. A separate checklist file is not required, but the project rubric gives points for performing **and summarizing** light validation checks, and a checklist is a simple way to write that summary.

### What a validation checklist is

A validation checklist gathers the validation code you already ran into one table: one row per check, with the number the check found, a status, and a short note. In this lab, the checks from Sections 2-6 become these rows:

| Earlier check | Checklist row |
|---|---|
| Section 2: duplicate check on `origin` and `dest` | Duplicate route keys |
| Section 2: missing values in `origin` and `dest` | Missing key values |
| Section 3: missing summary | Missing ABT cells |
| Section 4: impossible-value checks | Nonpositive flight counts, distances and air times |
| Section 4 and Your Turn 3: negative maximum delay | Routes with a negative `max_arr_delay` |
| Section 5: target check | Missing target values |
| Section 6: data dictionary | ABT columns without a dictionary row; blank dictionary cells |

Each row gets one status:

- `Pass`: the check found no problem (the result is 0).
- `Documented`: the check found something real, and the data dictionary notes explain it.
- `Needs attention`: a problem that must be fixed or explained before handoff.

The cell below builds the first two rows. The result is computed by the same code you ran in Section 2; the status and the note are written by you. The offline assignment completes the checklist.

In [24]:
checklist_rows = [
    {
        "check": "Duplicate route keys (origin + dest)",
        "result": route_abt[["origin", "dest"]].duplicated().sum(),
        "status": "Pass",
        "notes": "Each route appears once",
    },
    {
        "check": "Missing key values (origin, dest)",
        "result": route_abt[["origin", "dest"]].isna().sum().sum(),
        "status": "Pass",
        "notes": "Every row has both key values",
    },
]

pd.DataFrame(checklist_rows)

,check,result,status,notes
0,Duplicate route keys (origin + dest),0,Pass,Each route appears once
1,"Missing key values (origin, dest)",0,Pass,Every row has both key values


## 9. Offline Assignment - Handoff Package

An analyst on another team will use the route ABT next week and cannot ask you to explain undocumented details. Finish the handoff package and decide whether it is ready.

1. Complete the data dictionary rows for the five ABT columns that are still blank.
2. Complete the validation checklist.
3. Run the export cell to save the three handoff files.
4. Write the handoff decision. A real extreme delay can remain if it is documented; do not silently remove it.

### Step 1: Finish the data dictionary

In [25]:
# Offline Assignment, step 1 - document the five remaining ABT columns.
# Replace the empty strings, four values per column in the order of doc_fields, then run the cell.
remaining_entries = {
    "max_arr_delay": ["", "", "", ""],
    "avg_distance": ["", "", "", ""],
    "avg_air_time": ["", "", "", ""],
    "dest_lat": ["", "", "", ""],
    "dest_lon": ["", "", "", ""],
}

for abt_column, values in remaining_entries.items():
    data_dictionary.loc[data_dictionary["column_name"] == abt_column, doc_fields] = values

data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,origin,str,Origin airport code for the route,flights table,Used as part of the route-level grouping key,Forms the route key together with dest; only t...
1,dest,str,Destination airport code for the route,flights table,Used as part of the route-level grouping key,"Includes one unusual route, EWR to LGA, with a..."
2,flight_count,int64,Number of flights observed for the route,flights table,Count of flight records grouped by origin and ...,Includes cancelled flights; ranges from 1 to 1...
3,avg_dep_delay,float64,,,,
4,avg_arr_delay,float64,Average arrival delay for flights on the route...,flights table,Mean of arr_delay grouped by origin and destin...,Averages only flights with a recorded arrival ...
5,max_arr_delay,float64,,,,
6,avg_distance,float64,,,,
7,avg_air_time,float64,,,,
8,dest_airport_name,str,Full name of the destination airport,airports table,Joined from airports using dest = faa,"Missing for 7 routes whose destination (BQN, P..."
9,dest_lat,float64,,,,


### Step 2: Complete the validation checklist

The first two rows come from Section 8. For each new row:

1. Replace `None` in `result` with code that computes the number. Reuse your code from Sections 2-6.
2. Replace `TODO` in `status` with `Pass`, `Documented` or `Needs attention`, using the definitions in Section 8.
3. Replace `TODO` in `notes` with a short note: what the result means, and where it is documented when the status is `Documented`.

In [26]:
# Offline Assignment, step 2 - complete the validation checklist.
# Hints:
# - Missing ABT cells: .isna().sum().sum() on route_abt.
# - Nonpositive values: add up the three Boolean checks from Section 4.
# - Negative max_arr_delay: the condition from Your Turn 3, then .sum().
# - ABT columns without a dictionary row: compare route_abt.columns with data_dictionary["column_name"].
# - Blank dictionary cells: select doc_fields, .fillna(""), strip whitespace, compare to "", then sum twice.
validation_checklist = pd.DataFrame(checklist_rows + [
    {"check": "Missing ABT cells (all columns)", "result": None, "status": "TODO", "notes": "TODO"},
    {"check": "Nonpositive flight counts, distances and air times", "result": None, "status": "TODO", "notes": "TODO"},
    {"check": "Routes with a negative max_arr_delay", "result": None, "status": "TODO", "notes": "TODO"},
    {"check": "Missing target values (avg_arr_delay)", "result": None, "status": "TODO", "notes": "TODO"},
    {"check": "ABT columns without a dictionary row", "result": None, "status": "TODO", "notes": "TODO"},
    {"check": "Blank data dictionary cells", "result": None, "status": "TODO", "notes": "TODO"},
])

validation_checklist

,check,result,status,notes
0,Duplicate route keys (origin + dest),0.0,Pass,Each route appears once
1,"Missing key values (origin, dest)",0.0,Pass,Every row has both key values
2,Missing ABT cells (all columns),NaN,TODO,TODO
3,"Nonpositive flight counts, distances and air t...",NaN,TODO,TODO
4,Routes with a negative max_arr_delay,NaN,TODO,TODO
5,Missing target values (avg_arr_delay),NaN,TODO,TODO
6,ABT columns without a dictionary row,NaN,TODO,TODO
7,Blank data dictionary cells,NaN,TODO,TODO


### Step 3: Export the handoff files

In [27]:
# Offline Assignment, step 3 - export the three handoff files. Run this after the checklist cell.
unfinished = validation_checklist["result"].isna() | validation_checklist[["status", "notes"]].eq("TODO").any(axis=1)
if unfinished.any():
    print("Complete these checklist rows before exporting:", list(validation_checklist.loc[unfinished, "check"]))
else:
    handoff_files = {
        "route_abt_module11.csv": route_abt,
        "route_abt_data_dictionary_module11.csv": data_dictionary,
        "route_abt_validation_checklist_module11.csv": validation_checklist,
    }
    for file_name, table in handoff_files.items():
        table.to_csv(output_dir / file_name, index=False)
        print("Saved:", output_dir / file_name)

Complete these checklist rows before exporting: ['Missing ABT cells (all columns)', 'Nonpositive flight counts, distances and air times', 'Routes with a negative max_arr_delay', 'Missing target values (avg_arr_delay)', 'ABT columns without a dictionary row', 'Blank data dictionary cells']


### Step 4: Handoff Decision

**Your response here:** Write 5-6 sentences that approve the handoff, approve it with conditions, or hold it for revision. Cite at least four results from your validation checklist, identify one remaining limitation or target concern, and state what the receiving analyst should know before using the ABT for modeling.


## 10. Apply This to Your Project (after the lab)

This section is not part of the lab grade. Use the same process when you prepare your project ABT for the next milestone:

1. A clear statement of your ABT unit of analysis.
2. A summary of your validation checks (a checklist like the one in Section 9 works well).
3. A data dictionary for your ABT.
4. A transformation and integration summary.
5. A clean handoff folder in GitHub.

This work prepares you for the next module, where we begin applying analytics to the ABT.

## 11. Final Reflection

Write 3-4 sentences about your project ABT. Answer from what you know now; you do not need to build anything:

1. What does one row mean, and which columns are the key?
2. What is your likely target, and is it known for every row?
3. Which check from this lab will you run first on your project ABT, and which column will need the clearest dictionary entry?

**Your reflection:**  
Type your answer here.

## 12. Save Your Work

Before submitting:

1. Complete all Your Turn sections and the offline assignment (data dictionary, validation checklist).
2. Complete the handoff decision and final reflection.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm all three Module 11 files exist in `outputs`.
5. Save the notebook, then commit and push your work.

```bash
git add .
git commit -m "Complete Module 11 ABT documentation lab"
git push
```


## Optional Practice: Diagnose a Mini Handoff Package (Not Graded)

This short practice does **not** count toward your lab grade. Use it if you want another repetition of the Week 11 handoff-readiness workflow before applying the same checks to your project.

A teammate sends a small route ABT and data dictionary. Do not repair the files immediately. First build a validation checklist that shows whether the package is ready for another analyst.


In [28]:
practice_abt = pd.DataFrame({
    "origin": ["JFK", "JFK", "JFK", "LGA"],
    "dest": ["ORD", "ATL", "ATL", "BOS"],
    "flight_count": [820, 760, 760, 640],
    "avg_arr_delay": [8.4, 11.2, 11.2, pd.NA],
})

practice_dictionary = pd.DataFrame({
    "column_name": ["origin", "dest", "flight_count", "avg_arr_delay"],
    "description": ["Origin airport code", "Destination airport code", "Number of flights", ""],
    "source": ["flights.origin", "flights.dest", "flights", "flights.arr_delay"],
    "transformation_or_derivation": ["Grouped field", "Grouped field", "COUNT(*)", "AVG(arr_delay)"],
    "notes_or_concerns": ["Part of route key", "Part of route key", "Should be positive", "May be missing"],
})

display(practice_abt)
display(practice_dictionary)


,origin,dest,flight_count,avg_arr_delay
0,JFK,ORD,820,8.4
1,JFK,ATL,760,11.2
2,JFK,ATL,760,11.2
3,LGA,BOS,640,<NA>


,column_name,description,source,transformation_or_derivation,notes_or_concerns
0,origin,Origin airport code,flights.origin,Grouped field,Part of route key
1,dest,Destination airport code,flights.dest,Grouped field,Part of route key
2,flight_count,Number of flights,flights,COUNT(*),Should be positive
3,avg_arr_delay,,flights.arr_delay,AVG(arr_delay),May be missing


### Practice Task

Create a validation checklist named `practice_checklist` with the columns `check` and `result` and one row for each check:

1. Duplicate `origin`-`dest` keys.
2. Missing key values.
3. Missing `avg_arr_delay` target values.
4. ABT columns without a dictionary row.
5. Blank cells across `description`, `source`, `transformation_or_derivation` and `notes_or_concerns`.

Hints:

- Use `.duplicated(["origin", "dest"]).sum()` for the composite key.
- Select both key columns before applying `.isna().sum().sum()`.
- For documentation blanks, select the four text fields, apply `.fillna("")`, convert to string, strip whitespace, compare with `""`, and sum twice.
- Checklist first. A handoff decision should describe the evidence before recommending repairs.

In [29]:
# Optional practice workspace
# Replace None with a DataFrame that has the columns check and result, one row per requested check.
practice_checklist = None


In [30]:
# Run this self-check after you create practice_checklist.
if practice_checklist is None:
    print("Complete the optional practice first, then rerun this self-check.")
else:
    expected_practice_checklist = pd.DataFrame({
        "check": ["Duplicate route keys", "Missing key values", "Missing target values",
                  "ABT columns without a dictionary row", "Blank dictionary cells"],
        "result": [1, 0, 1, 0, 1],
    })
    print("Your checklist:")
    display(practice_checklist)
    print("Expected results:")
    display(expected_practice_checklist)


Complete the optional practice first, then rerun this self-check.


### Think About the Result

This package should be held for revision: it contains a duplicate route key, a missing target value, and an undocumented dictionary cell. Notice that every ABT column has a dictionary row, but one dictionary cell is still blank. A useful handoff decision should name each issue, explain why it matters to the receiving analyst, and request a specific correction or documented exception.
